# Notebook for clustering.

Run interactive notebooks in the following order:\
preprocessing.ipynb  ->  plot_corr_assoc.ipynb  ->  dimensionality_reduction.ipynb  ->  clustering.ipynb

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, HDBSCAN, OPTICS, KMeans  # type: ignore
from sklearn.mixture import BayesianGaussianMixture
from sklearn.metrics import rand_score, adjusted_rand_score
from sklearn.cluster import AgglomerativeClustering, AffinityPropagation
from umap import UMAP

from utils.clustering_utils import plot_data_2d, plot_silhouette, print_clustering_info
from utils.utils import check_save_path, load_data, save_data

pd.options.mode.copy_on_write = True

In [ ]:
# Create RNG for reproducibility
rng = np.random.default_rng(394325)

# Hardcoded paths, change to apply to new data
data_path = Path("outputs/dim_red/esa_2018_2021_reduced.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")

image_folder = "outputs/clustering"
plot_save_name_template = "clustering_plot_{}.png"

allow_overwrite = True

# Read in data
data: pd.DataFrame = load_data(data_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# # Normalize data
# from sklearn.preprocessing import StandardScaler
# standard_scaler = StandardScaler()
# data_transformed = standard_scaler.fit_transform(data)

# Create distance matrix
# from scipy.spatial.distance import pdist, squareform
# distances = pdist(data.values, metric='euclidean')
# dist_matrix = squareform(distances)
# ...

In [ ]:
# Plot projected data in 2d
rng = np.random.default_rng(23523)

projection_method = "UMAP"  # projection_method="TSNE"
data_plot = plot_data_2d(data=data, projection_method=projection_method, rng=rng)
data_plot.show()
data_plot_save_path = Path(image_folder, f"data_plot_{projection_method}.png")
check_save_path(data_plot_save_path, allow_overwrite=allow_overwrite)
data_plot.save(data_plot_save_path)
print(f"Saved plot of data projected by {projection_method} as PNG to {data_plot_save_path}.")

### k-means clustering with silhouette plots

In [ ]:
rng = np.random.default_rng(5252)

kmeans_n_clusters_list = [4, 5, 6, 7]
kmeans_clustering_list = []
kmeans_cluster_labels_list = []
kmeans_silhouette_plot_list = []
for n_clusters in kmeans_n_clusters_list:
    clustering = KMeans(n_clusters=n_clusters, random_state=np.random.RandomState(rng.bit_generator))
    cluster_labels = pd.Series(clustering.fit_predict(data)).replace(-1, pd.NA).astype("category")

    # Print information about clustering
    print_clustering_info(cluster_labels)

    # Create silhouette plot
    silhouette_plot = plot_silhouette(data, cluster_labels)

    # Append to list
    kmeans_clustering_list.append(clustering)
    kmeans_cluster_labels_list.append(cluster_labels)
    kmeans_silhouette_plot_list.append(silhouette_plot)

p1, p2, p3, p4 = kmeans_silhouette_plot_list
kmeans_silhouette_plots = (p1 | p2) / (p3 | p4)
# Save plot
kmeans_silhouette_plots_save_path = Path(image_folder, "kmeans_silhouette_plots.png")
check_save_path(kmeans_silhouette_plots_save_path, allow_overwrite=allow_overwrite)
kmeans_silhouette_plots.save(kmeans_silhouette_plots_save_path)
print(f"Saved K-means silhouette plots as PNG to {kmeans_silhouette_plots_save_path}.")
# Show plot
kmeans_silhouette_plots.draw()

### Affinity propagation clustering

In [ ]:
rng = np.random.default_rng(34522)

ap = AffinityPropagation(damping=0.5, preference=None, random_state=np.random.RandomState(rng.bit_generator))

ap_cluster_labels = pd.Series(ap.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(ap_cluster_labels)

-> crashes

### Agglomerative clustering

In [ ]:
ward = AgglomerativeClustering(n_clusters=10, linkage="ward")

ward_cluster_labels = pd.Series(ward.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(ward_cluster_labels)

-> crashes

### DBSCAN, OPTICS, and HDBSCAN clustering

In [ ]:
rng = np.random.default_rng(9458)

# dbscan_clustering = DBSCAN(eps=6.5, min_samples=10, n_jobs=-1)
# gmm_clustering = OPTICS(min_samples=5, n_jobs=-1)
dbscan_clustering = HDBSCAN(
    min_cluster_size=10,
    min_samples=1,
    # cluster_selection_epsilon=1.0,
    store_centers=None,
    n_jobs=-1,
)

dbscan_cluster_labels = pd.Series(dbscan_clustering.fit_predict(data)).replace(-1, pd.NA).astype("category")

# dbscan_cluster_probs = dbscan_clustering.probabilities_  # HDBSCAN, ...
# dbscan_cluster_centroids = dbscan_clustering.centroids_  # HDBSCAN with store_centers="both"
# dbscan_cluster_medoids = dbscan_clustering.medoids_  # HDBSCAN with store_centers="both"

# Print information about clustering
print_clustering_info(dbscan_cluster_labels)

In [ ]:
# Plot clustering
projection_method = "UMAP"  # projection_method="TSNE"
dbscan_clustering_plot = plot_data_2d(
    data=data, cluster_labels=dbscan_cluster_labels, projection_method=projection_method, rng=rng
)
# Save plot
dbscan_clustering_plot_save_path = Path(image_folder, "dbscan_clustering_plot.png")
check_save_path(dbscan_clustering_plot_save_path, allow_overwrite=allow_overwrite)
dbscan_clustering_plot.save(dbscan_clustering_plot_save_path)
print(
    f"Saved plot of HDBSCAN clustering projected by {projection_method} as PNG to {dbscan_clustering_plot_save_path}."
)
# Show plot
dbscan_clustering_plot.show()

### Gaussian mixture model clustering with Dirichlet (process) prior

In [ ]:
rng = np.random.default_rng(2352)

gmm = BayesianGaussianMixture(
    n_components=10,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    weight_concentration_prior=1e3,
    mean_precision_prior=1.0,
    random_state=np.random.RandomState(rng.bit_generator),
)

gmm_cluster_labels = pd.Series(gmm.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(gmm_cluster_labels)

In [ ]:
# Silhouette plot
gmm_silhouette_plot = plot_silhouette(data, gmm_cluster_labels)
# Save plot
gmm_silhouette_plot_save_path = Path(image_folder, "gmm_silhouette_plot.png")
check_save_path(gmm_silhouette_plot_save_path, allow_overwrite=allow_overwrite)
gmm_silhouette_plot.save(gmm_silhouette_plot_save_path)
print(f"Saved GMM silhouette plot as PNG to {gmm_silhouette_plot_save_path}.")
# Show plot
gmm_silhouette_plot.show()

In [ ]:
# Plot clustering
projection_method = "UMAP"  # projection_method="TSNE"
gmm_clustering_plot = plot_data_2d(
    data=data, cluster_labels=gmm_cluster_labels, projection_method=projection_method, rng=rng
)
# Save plot
gmm_clustering_plot_save_path = Path(image_folder, "gmm_clustering_plot.png")
check_save_path(gmm_clustering_plot_save_path, allow_overwrite=allow_overwrite)
gmm_clustering_plot.save(gmm_clustering_plot_save_path)
print(f"Saved plot of GMM clustering projected by {projection_method} as PNG to {gmm_clustering_plot_save_path}.")
# Show plot
gmm_clustering_plot.show()

### Compare clustering results

In [ ]:
na_mask = dbscan_cluster_labels.isnull()
dbscan_gmm_rand_score = rand_score(dbscan_cluster_labels[~ na_mask], gmm_cluster_labels[~ na_mask])
dbscan_gmm_rand_score

In [ ]:
na_mask = dbscan_cluster_labels.isnull()
dbscan_gmm_adjusted_rand_score = adjusted_rand_score(dbscan_cluster_labels[~ na_mask], gmm_cluster_labels[~ na_mask])
dbscan_gmm_adjusted_rand_score